# this work — Fase 3: Análise Completa (5M tokens por corpus, 20M total, dois níveis)

**Interpretabilidade de Modelos de Linguagem em Português via Sparse Autoencoders**

---

**Objetivo:** Escalar a análise piloto para o experimento completo.

**O que este notebook faz:**
1. Processa **5M tokens por corpus** (4 corpora = 20M tokens total)
2. **Dois níveis de comparação:**
   - Nível 1 (Web-crawl): FineWeb-2 PT vs FineWeb EN
   - Nível 2 (Wikipedia): Wikipedia PT vs Wikipedia EN
3. Calcula **LSI em ambos os níveis** para triangulação
4. Seleciona **150 features** para análise manual (50 PT + 50 cross + 50 EN)
5. Extrai **top-20 exemplos ativadores** por feature
6. Gera visualizações completas

**Pré-requisitos:**
- GPU com ≥16GB VRAM (T4 no Colab)
- [Aceitar licença do Gemma 2](https://huggingface.co/google/gemma-2-2b)
- Tempo estimado: ~2-3 horas no T4

In [1]:
!pip install sae-lens transformer-lens datasets -q
print()
print("Instalação concluída. Reinicie o runtime antes de continuar:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Depois, pule esta célula e execute a partir da próxima.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./data/checkpoints/ for Colab)
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('Sem GPU detectada. Processamento será muito lento.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4 (15.6 GB)
Device: cuda
PyTorch: 2.10.0+cu128


In [3]:
from huggingface_hub import login
login()

In [4]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Carregando SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

Carregando Gemma 2 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Modelo: gemma-2-2b | Layers: 26 | d_model: 2304

Carregando SAE: layer_13/width_16k/canonical...


layer_13/width_16k/average_l0_84/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

SAE: 16384 features | d_in: 2304 | hook: blocks.13.hook_resid_post


/tmp/ipython-input-861/3908203106.py:17: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


---
# Fase 3: Coleta de Dados (4 corpora)

Coletamos tokens de **4 fontes** para comparação em dois níveis:

| Nível | Corpus PT | Corpus EN | Propósito |
|---|---|---|---|
| **1 — Web-crawl** | FineWeb-2 PT | FineWeb EN | Comparação realista (pode ter confounders de domínio) |
| **2 — Wikipedia** | Wikipedia PT | Wikipedia EN | Comparação controlada (isola efeito de língua) |

**Triangulação:** features PT-específicas em **ambos os níveis** = forte evidência linguística.

In [5]:
N_TOKENS = 5_000_000     # 5M tokens por corpus (ajuste se necessário)
SEQ_LEN = 256
BATCH_SIZE = 8
MIN_ACTS = 100           # ativações mínimas para considerar feature ativa
LSI_THRESHOLD = 0.3      # limiar para classificar como language-specific
N_SELECT = 50            # features por categoria (PT, EN, cross-lingual)
N_EXAMPLES = 20          # exemplos ativadores por feature

n_seqs = N_TOKENS // SEQ_LEN
n_batches = n_seqs // BATCH_SIZE

print(f"Tokens por corpus: {N_TOKENS:,}")
print(f"Total de tokens (4 corpora): {N_TOKENS * 4:,}")
print(f"Sequências por corpus: {n_seqs:,} ({SEQ_LEN} tokens cada)")
print(f"Batches por corpus: {n_batches:,} (batch size {BATCH_SIZE})")
print(f"Features selecionadas: {N_SELECT * 3} (3 × {N_SELECT})")

Tokens por corpus: 5,000,000
Total de tokens (4 corpora): 20,000,000
Sequências por corpus: 19,531 (256 tokens cada)
Batches por corpus: 2,441 (batch size 8)
Features selecionadas: 150 (3 × 50)


In [6]:
from datasets import load_dataset

tokenizer = model.tokenizer

def collect_tokens(dataset, n_tokens, text_field="text", desc="Tokenizando"):
    all_ids = []
    n_articles = 0
    for article in tqdm(dataset, desc=desc):
        text = article[text_field]
        if not text or len(text) < 50:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_ids.extend(ids)
        n_articles += 1
        if len(all_ids) >= n_tokens:
            break
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_articles} textos -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs)")
    return tokens

print("=" * 60)
print("NÍVEL 2: WIKIPEDIA (comparação controlada)")
print("=" * 60)

print("\nCarregando Wikipedia PT...")
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
tokens_wiki_pt = collect_tokens(wiki_pt, N_TOKENS, desc="Wiki PT")

print("\nCarregando Wikipedia EN...")
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
tokens_wiki_en = collect_tokens(wiki_en, N_TOKENS, desc="Wiki EN")

print()
print("=" * 60)
print("NÍVEL 1: WEB-CRAWL (comparação realista)")
print("=" * 60)

print("\nCarregando FineWeb-2 PT...")
web_pt = load_dataset("HuggingFaceFW/fineweb-2", "por_Latn", split="train", streaming=True)
tokens_mc4_pt = collect_tokens(web_pt, N_TOKENS, desc="FineWeb PT")

print("\nCarregando FineWeb EN...")
web_en = load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True)
tokens_c4_en = collect_tokens(web_en, N_TOKENS, desc="FineWeb EN")

print()
print("Coleta concluída!")
print(f"Total: {(tokens_wiki_pt.numel() + tokens_wiki_en.numel() + tokens_mc4_pt.numel() + tokens_c4_en.numel()):,} tokens")

NÍVEL 2: WIKIPEDIA (comparação controlada)

Carregando Wikipedia PT...


README.md: 0.00B [00:00, ?B/s]

Wiki PT: 0it [00:00, ?it/s]

  994 textos -> 4,999,936 tokens (19531 seqs)

Carregando Wikipedia EN...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

...

  5674 textos -> 4,999,936 tokens (19531 seqs)

Carregando FineWeb EN...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

FineWeb EN: 0it [00:00, ?it/s]

  7291 textos -> 4,999,936 tokens (19531 seqs)

Coleta concluída!
Total: 19,999,744 tokens


---
# Fase 3: Processamento de Ativações

Passamos todos os tokens pelo **Gemma 2 2B** e extraímos ativações da **layer 13**, codificando com o **SAE de 16k features**.

Tempo estimado: ~30 min por corpus no T4 (~2h total).

In [7]:
@torch.no_grad()
def compute_feature_stats(model, sae, tokens, batch_size=BATCH_SIZE, desc=""):
    n_features = sae.cfg.d_sae
    hook = HOOK_NAME
    counts = torch.zeros(n_features, device=device)
    sums = torch.zeros(n_features, device=device)
    maxvals = torch.zeros(n_features, device=device)
    total = 0

    n_batches = (len(tokens) + batch_size - 1) // batch_size
    t0 = time.time()
    for i in tqdm(range(0, len(tokens), batch_size), desc=desc, total=n_batches):
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        active = feat_acts > 0
        counts += active.float().sum(dim=(0, 1))
        sums += feat_acts.sum(dim=(0, 1))
        maxvals = torch.max(maxvals, feat_acts.amax(dim=(0, 1)))
        total += batch.numel()

        del cache, acts, feat_acts, active
        if device == "cuda":
            torch.cuda.empty_cache()

    elapsed = time.time() - t0
    print(f"  {total:,} tokens | {(counts > 0).sum().item():,} features ativas | {elapsed:.0f}s")
    return {"counts": counts.cpu(), "sums": sums.cpu(), "max": maxvals.cpu(), "total_tokens": total}

In [8]:
import os

# --- Wiki PT ---
if os.path.exists("stats_wiki_pt.pt"):
    print("Carregando checkpoint stats_wiki_pt.pt...")
    stats_wiki_pt = torch.load("stats_wiki_pt.pt")
else:
    print("Wikipedia PT...")
    stats_wiki_pt = compute_feature_stats(model, sae, tokens_wiki_pt, desc="Wiki PT")
    torch.save(stats_wiki_pt, SAVE_DIR + "stats_wiki_pt.pt")
    print("Checkpoint salvo: stats_wiki_pt.pt")

# --- Wiki EN ---
if os.path.exists("stats_wiki_en.pt"):
    print("Carregando checkpoint stats_wiki_en.pt...")
    stats_wiki_en = torch.load("stats_wiki_en.pt")
else:
    print("\nWikipedia EN...")
    stats_wiki_en = compute_feature_stats(model, sae, tokens_wiki_en, desc="Wiki EN")
    torch.save(stats_wiki_en, SAVE_DIR + "stats_wiki_en.pt")
    print("Checkpoint salvo: stats_wiki_en.pt")

Wikipedia PT...


Wiki PT:   0%|          | 0/2442 [00:00<?, ?it/s]

  4,999,936 tokens | 16,217 features ativas | 2342s
Checkpoint salvo: stats_wiki_pt.pt

Wikipedia EN...


Wiki EN:   0%|          | 0/2442 [00:00<?, ?it/s]

  4,999,936 tokens | 16,305 features ativas | 2341s
Checkpoint salvo: stats_wiki_en.pt


In [9]:
import os

# --- FineWeb PT ---
if os.path.exists("stats_mc4_pt.pt"):
    print("Carregando checkpoint stats_mc4_pt.pt...")
    stats_mc4_pt = torch.load("stats_mc4_pt.pt")
else:
    print("FineWeb PT...")
    stats_mc4_pt = compute_feature_stats(model, sae, tokens_mc4_pt, desc="FineWeb PT")
    torch.save(stats_mc4_pt, SAVE_DIR + "stats_mc4_pt.pt")
    print("Checkpoint salvo: stats_mc4_pt.pt")

# --- FineWeb EN ---
if os.path.exists("stats_c4_en.pt"):
    print("Carregando checkpoint stats_c4_en.pt...")
    stats_c4_en = torch.load("stats_c4_en.pt")
else:
    print("\nFineWeb EN...")
    stats_c4_en = compute_feature_stats(model, sae, tokens_c4_en, desc="FineWeb EN")
    torch.save(stats_c4_en, SAVE_DIR + "stats_c4_en.pt")
    print("Checkpoint salvo: stats_c4_en.pt")

FineWeb PT...


FineWeb PT:   0%|          | 0/2442 [00:00<?, ?it/s]

  4,999,936 tokens | 16,203 features ativas | 2341s
Checkpoint salvo: stats_mc4_pt.pt

FineWeb EN...


FineWeb EN:   0%|          | 0/2442 [00:00<?, ?it/s]

  4,999,936 tokens | 16,287 features ativas | 2341s
Checkpoint salvo: stats_c4_en.pt


---
# Fase 3: Análise — LSI em Dois Níveis + Seleção de Features

Calculamos o LSI separadamente para cada nível e usamos **triangulação** para distinguir efeitos linguísticos de efeitos de domínio.

In [10]:
# --- LSI Nível 2: Wikipedia ---
freq_wiki_pt = stats_wiki_pt["counts"] / stats_wiki_pt["total_tokens"]
freq_wiki_en = stats_wiki_en["counts"] / stats_wiki_en["total_tokens"]
lsi_wiki = (freq_wiki_pt - freq_wiki_en) / (freq_wiki_pt + freq_wiki_en + 1e-10)

# --- LSI Nível 1: Web-crawl ---
freq_mc4_pt = stats_mc4_pt["counts"] / stats_mc4_pt["total_tokens"]
freq_c4_en = stats_c4_en["counts"] / stats_c4_en["total_tokens"]
lsi_web = (freq_mc4_pt - freq_c4_en) / (freq_mc4_pt + freq_c4_en + 1e-10)

# --- Features ativas ---
total_counts = (stats_wiki_pt["counts"] + stats_wiki_en["counts"]
                + stats_mc4_pt["counts"] + stats_c4_en["counts"])
alive = total_counts > 0
active = total_counts >= MIN_ACTS

n_total = sae.cfg.d_sae
n_dead = (~alive).sum().item()
n_alive = alive.sum().item()
n_active = active.sum().item()

# --- Triangulação ---
pt_wiki = (lsi_wiki > LSI_THRESHOLD) & active
pt_web = (lsi_web > LSI_THRESHOLD) & active
pt_both = pt_wiki & pt_web   # forte evidência linguística
pt_wiki_only = pt_wiki & ~pt_web
pt_web_only = pt_web & ~pt_wiki

en_wiki = (lsi_wiki < -LSI_THRESHOLD) & active
en_web = (lsi_web < -LSI_THRESHOLD) & active
en_both = en_wiki & en_web

cross_wiki = (lsi_wiki.abs() <= LSI_THRESHOLD) & active
cross_web = (lsi_web.abs() <= LSI_THRESHOLD) & active
cross_both = cross_wiki & cross_web

print("=" * 65)
print("ESTATÍSTICAS GERAIS")
print("=" * 65)
print(f"Total features:              {n_total:,}")
print(f"Features mortas:             {n_dead:,} ({n_dead/n_total*100:.1f}%)")
print(f"Features vivas:              {n_alive:,} ({n_alive/n_total*100:.1f}%)")
print(f"Features ativas (>={MIN_ACTS}):     {n_active:,}")

print()
print("=" * 65)
print("LSI — NÍVEL 2 (WIKIPEDIA)")
print("=" * 65)
lsi_w = lsi_wiki[active]
print(f"  Média:    {lsi_w.mean().item():.4f}")
print(f"  Mediana:  {lsi_w.median().item():.4f}")
print(f"  PT-predominantes (>{LSI_THRESHOLD}):  {pt_wiki.sum().item()}")
print(f"  EN-predominantes (<{-LSI_THRESHOLD}): {en_wiki.sum().item()}")
print(f"  Cross-lingual:              {cross_wiki.sum().item()}")

print()
print("=" * 65)
print("LSI — NÍVEL 1 (WEB-CRAWL)")
print("=" * 65)
lsi_wc = lsi_web[active]
print(f"  Média:    {lsi_wc.mean().item():.4f}")
print(f"  Mediana:  {lsi_wc.median().item():.4f}")
print(f"  PT-predominantes (>{LSI_THRESHOLD}):  {pt_web.sum().item()}")
print(f"  EN-predominantes (<{-LSI_THRESHOLD}): {en_web.sum().item()}")
print(f"  Cross-lingual:              {cross_web.sum().item()}")

print()
print("=" * 65)
print("TRIANGULAÇÃO")
print("=" * 65)
print(f"  PT em AMBOS os níveis:     {pt_both.sum().item()}  ← forte evidência linguística")
print(f"  PT só Wikipedia:           {pt_wiki_only.sum().item()}  ← possível efeito de estilo")
print(f"  PT só Web-crawl:           {pt_web_only.sum().item()}  ← possível efeito de domínio")
print(f"  EN em AMBOS os níveis:     {en_both.sum().item()}")
print(f"  Cross-lingual em ambos:    {cross_both.sum().item()}")

ESTATÍSTICAS GERAIS
Total features:              16,384
Features mortas:             35 (0.2%)
Features vivas:              16,349 (99.8%)
Features ativas (>=100):     15,703

LSI — NÍVEL 2 (WIKIPEDIA)
  Média:    -0.3601
  Mediana:  -0.4004
  PT-predominantes (>0.3):  914
  EN-predominantes (<-0.3): 9117
  Cross-lingual:              5672

LSI — NÍVEL 1 (WEB-CRAWL)
  Média:    -0.3566
  Mediana:  -0.4008
  PT-predominantes (>0.3):  1039
  EN-predominantes (<-0.3): 9088
  Cross-lingual:              5576

TRIANGULAÇÃO
  PT em AMBOS os níveis:     674  ← forte evidência linguística
  PT só Wikipedia:           240  ← possível efeito de estilo
  PT só Web-crawl:           365  ← possível efeito de domínio
  EN em AMBOS os níveis:     8212
  Cross-lingual em ambos:    4482


In [11]:
# --- LSI combinado (média dos dois níveis) ---
lsi_combined = (lsi_wiki + lsi_web) / 2

# --- Seleção: Top-50 PT ---
lsi_pt_rank = lsi_combined.clone()
lsi_pt_rank[~active] = -2
top_pt_vals, top_pt_idx = lsi_pt_rank.topk(N_SELECT)

# --- Seleção: Top-50 EN ---
lsi_en_rank = lsi_combined.clone()
lsi_en_rank[~active] = 2
top_en_vals, top_en_idx = lsi_en_rank.topk(N_SELECT, largest=False)

# --- Seleção: Top-50 Cross-lingual (|LSI| mais próximo de 0, com alta frequência) ---
total_freq = freq_wiki_pt + freq_wiki_en + freq_mc4_pt + freq_c4_en
cross_score = lsi_combined.abs().clone()
cross_score[~active] = 999
min_cross_freq = total_freq[active].median()
cross_score[total_freq < min_cross_freq] = 999
top_cross_vals, top_cross_idx = cross_score.topk(N_SELECT, largest=False)

# --- Consolidar ---
selected_pt = top_pt_idx.tolist()
selected_en = top_en_idx.tolist()
selected_cross = top_cross_idx.tolist()
all_selected = selected_pt + selected_cross + selected_en

print("=" * 90)
print(f"FEATURES SELECIONADAS: {len(all_selected)} total")
print("=" * 90)

def print_table(label, indices, lsi_tensor):
    fmt = '{:<4} {:<8} {:<10} {:<10} {:<10} {:<10} {:<10}'
    print(f"\n{'—' * 90}")
    print(f"{label} ({len(indices)} features)")
    print('—' * 90)
    print(fmt.format('#', 'Feature', 'LSI Wiki', 'LSI Web', 'LSI Comb', 'Freq PT*', 'Freq EN*'))
    print('-' * 90)
    for i, idx in enumerate(indices[:20]):
        freq_pt_total = (freq_wiki_pt[idx] + freq_mc4_pt[idx]).item() / 2
        freq_en_total = (freq_wiki_en[idx] + freq_c4_en[idx]).item() / 2
        print(fmt.format(
            i+1, idx,
            f'{lsi_wiki[idx].item():+.4f}',
            f'{lsi_web[idx].item():+.4f}',
            f'{lsi_combined[idx].item():+.4f}',
            f'{freq_pt_total:.6f}',
            f'{freq_en_total:.6f}',
        ))
    if len(indices) > 20:
        print(f"  ... (+{len(indices) - 20} features não exibidas)")
    print("  * Freq = média dos dois corpora por idioma")

print_table("TOP PT-ESPECÍFICAS", selected_pt, lsi_combined)
print_table("TOP CROSS-LINGUAL", selected_cross, lsi_combined)
print_table("TOP EN-ESPECÍFICAS (controle)", selected_en, lsi_combined)

# Marcar quais são consistentes nos dois níveis
n_pt_consistent = sum(1 for idx in selected_pt if pt_both[idx].item())
print(f"\n>>> Das {N_SELECT} PT selecionadas, {n_pt_consistent} são consistentes em ambos os níveis.")

FEATURES SELECIONADAS: 150 total

——————————————————————————————————————————————————————————————————————————————————————————
TOP PT-ESPECÍFICAS (50 features)
——————————————————————————————————————————————————————————————————————————————————————————
#    Feature  LSI Wiki   LSI Web    LSI Comb   Freq PT*   Freq EN*  
------------------------------------------------------------------------------------------
1    10496    +0.9981    +0.9983    +0.9982    0.014697   0.000013  
2    1906     +0.9981    +0.9981    +0.9981    0.215058   0.000207  
3    8718     +0.9977    +0.9978    +0.9977    0.011960   0.000014  
4    16057    +0.9950    +0.9981    +0.9965    0.048321   0.000084  
5    4584     +0.9945    +0.9962    +0.9953    0.027500   0.000064  
6    2294     +0.9916    +0.9972    +0.9944    0.048532   0.000139  
7    15135    +0.9936    +0.9947    +0.9942    0.008714   0.000025  
8    12649    +0.9920    +0.9953    +0.9937    0.045004   0.000138  
9    10478    +0.9915    +0.9926    +0.

In [12]:
@torch.no_grad()
def find_top_examples(model, sae, tokens, feature_indices,
                      n_examples=N_EXAMPLES, batch_size=BATCH_SIZE, max_batches=500):
    hook = HOOK_NAME
    feat_list = list(feature_indices)
    results = {f: [] for f in feat_list}
    tok = model.tokenizer

    actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
    for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                  desc="Buscando exemplos", total=actual_batches):
        if i >= len(tokens):
            break
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        for f_idx in feat_list:
            vals = feat_acts[:, :, f_idx]
            for b in range(batch.shape[0]):
                max_val, max_pos = vals[b].max(dim=0)
                if max_val.item() > 0:
                    pos = max_pos.item()
                    start = max(0, pos - 10)
                    end = min(batch.shape[1], pos + 15)
                    ctx = tok.decode(batch[b, start:end].tolist())
                    results[f_idx].append({"act": max_val.item(), "context": ctx})

        del cache, acts, feat_acts
        if device == "cuda":
            torch.cuda.empty_cache()

    for f in results:
        results[f].sort(key=lambda x: x["act"], reverse=True)
        results[f] = results[f][:n_examples]
    return results

# Buscar exemplos PT (em Wiki PT, corpus mais limpo)
print("Buscando exemplos para features PT-específicas (Wiki PT)...")
examples_pt = find_top_examples(model, sae, tokens_wiki_pt, selected_pt)

# Buscar exemplos cross-lingual (em Wiki PT)
print("\nBuscando exemplos para features cross-lingual (Wiki PT)...")
examples_cross = find_top_examples(model, sae, tokens_wiki_pt, selected_cross)

# Buscar exemplos EN (em Wiki EN)
print("\nBuscando exemplos para features EN-específicas (Wiki EN)...")
examples_en = find_top_examples(model, sae, tokens_wiki_en, selected_en)

Buscando exemplos para features PT-específicas (Wiki PT)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]


Buscando exemplos para features cross-lingual (Wiki PT)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]


Buscando exemplos para features EN-específicas (Wiki EN)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]

In [13]:
print("=" * 80)
print("EXEMPLOS ATIVADORES — TOP-20 FEATURES PT-ESPECÍFICAS")
print("=" * 80)

for rank, f_idx in enumerate(selected_pt[:20]):
    print()
    print("-" * 70)
    consistent = "AMBOS NÍVEIS" if pt_both[f_idx].item() else "apenas 1 nível"
    print(f'Feature {f_idx} | LSI wiki={lsi_wiki[f_idx].item():+.4f} | '
          f'LSI web={lsi_web[f_idx].item():+.4f} | Rank #{rank+1} | {consistent}')
    print("-" * 70)
    exs = examples_pt.get(f_idx, [])
    if not exs:
        print("  (sem exemplos)")
        continue
    for ex in exs[:5]:
        print(f'  [{ex["act"]:.2f}] ...{ex["context"]}...')

EXEMPLOS ATIVADORES — TOP-20 FEATURES PT-ESPECÍFICAS

----------------------------------------------------------------------
Feature 10496 | LSI wiki=+0.9981 | LSI web=+0.9983 | Rank #1 | AMBOS NÍVEIS
----------------------------------------------------------------------
  [21.42] ... Aires).

 Economia 

A economia da Argentina é a terceira maior da América Latina, com uma alta qualidade de vida e...
  [20.83] ...os. Nas regiões do sul, os verões são mornos e invernos frios com fortes nevoadas, especialmente nas...
  [20.63] ...

Etimologia 
Em grego antigo, Atenas era chamada  (Athénai), em homenagem à deusa grega...
  [19.88] ... 156 km de Belo Horizonte. É considerada polo para algumas cidades de pequeno porte próximas tendo recebido em ...
  [19.79] ...ises racionais da sociedade, que a dominação é construída socialmente e que é injusta, e por isso,...

----------------------------------------------------------------------
Feature 1906 | LSI wiki=+0.9981 | LSI web=+0.9981 | Rank #

In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Histograma LSI Wikipedia
lsi_w_np = lsi_wiki[active].numpy()
axes[0, 0].hist(lsi_w_np, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 0].axvline(-LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 0].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 0].set_xlabel('LSI')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('(A) LSI Distribution — Wikipedia')

# (B) Histograma LSI Web-crawl
lsi_wc_np = lsi_web[active].numpy()
axes[0, 1].hist(lsi_wc_np, bins=100, color='darkorange', edgecolor='white', alpha=0.8)
axes[0, 1].axvline(LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 1].axvline(-LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].set_xlabel('LSI')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('(B) LSI Distribution — Web-crawl')

# (C) Triangulação: LSI Wiki vs LSI Web
lsi_w_a = lsi_wiki[active].numpy()
lsi_wc_a = lsi_web[active].numpy()
colors = np.where(
    (lsi_w_a > LSI_THRESHOLD) & (lsi_wc_a > LSI_THRESHOLD), 'red',
    np.where(
        (lsi_w_a < -LSI_THRESHOLD) & (lsi_wc_a < -LSI_THRESHOLD), 'blue',
        'gray'
    )
)
axes[1, 0].scatter(lsi_w_a, lsi_wc_a, c=colors, s=3, alpha=0.4)
axes[1, 0].axhline(LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axhline(-LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axvline(LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axvline(-LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].plot([-1, 1], [-1, 1], 'k--', linewidth=0.8, alpha=0.4)
axes[1, 0].set_xlabel('LSI (Wikipedia)')
axes[1, 0].set_ylabel('LSI (Web-crawl)')
axes[1, 0].set_title('(C) Triangulation: Wiki vs Web LSI')
axes[1, 0].set_xlim(-1.05, 1.05)
axes[1, 0].set_ylim(-1.05, 1.05)

# (D) Heatmap: Top-20 PT features, 4 corpora
top20_pt = selected_pt[:20]
data = np.column_stack([
    freq_wiki_pt[top20_pt].numpy(),
    freq_wiki_en[top20_pt].numpy(),
    freq_mc4_pt[top20_pt].numpy(),
    freq_c4_en[top20_pt].numpy(),
])
im = axes[1, 1].imshow(data, aspect='auto', cmap='YlOrRd')
axes[1, 1].set_xticks([0, 1, 2, 3])
axes[1, 1].set_xticklabels(['Wiki PT', 'Wiki EN', 'FineWeb PT', 'FineWeb EN'], fontsize=8)
axes[1, 1].set_yticks(range(len(top20_pt)))
labels = [f'F{idx}' for idx in top20_pt]
axes[1, 1].set_yticklabels(labels, fontsize=7)
axes[1, 1].set_title('(D) Top-20 PT Features: Freq by Corpus')
plt.colorbar(im, ax=axes[1, 1], label='Activation freq', shrink=0.8)

plt.tight_layout()
plt.savefig('fig_fase3_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig_fase3_overview.png")

Salvo: fig_fase3_overview.png


In [15]:
fig, ax = plt.subplots(figsize=(10, 16))

data50 = np.column_stack([
    freq_wiki_pt[selected_pt].numpy(),
    freq_wiki_en[selected_pt].numpy(),
    freq_mc4_pt[selected_pt].numpy(),
    freq_c4_en[selected_pt].numpy(),
])
im = ax.imshow(data50, aspect='auto', cmap='YlOrRd')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Wiki PT', 'Wiki EN', 'FineWeb PT', 'FineWeb EN'])
ax.set_yticks(range(len(selected_pt)))
ylabels = []
for idx in selected_pt:
    marker = "★" if pt_both[idx].item() else " "
    ylabels.append(f'{marker} F{idx} (LSI={lsi_combined[idx].item():+.2f})')
ax.set_yticklabels(ylabels, fontsize=6)
ax.set_title(f'Top-{N_SELECT} PT-Specific Features — Activation Frequency\n(★ = consistent across both levels)')
plt.colorbar(im, label='Activation frequency', shrink=0.6)
plt.tight_layout()
plt.savefig('fig_fase3_heatmap_top50_pt.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig_fase3_heatmap_top50_pt.png")

Salvo: fig_fase3_heatmap_top50_pt.png


In [16]:
results = {
    "config": {
        "n_tokens": N_TOKENS,
        "seq_len": SEQ_LEN,
        "layer": LAYER,
        "min_acts": MIN_ACTS,
        "lsi_threshold": LSI_THRESHOLD,
        "n_select": N_SELECT,
    },
    "stats": {
        "wiki_pt": stats_wiki_pt,
        "wiki_en": stats_wiki_en,
        "mc4_pt": stats_mc4_pt,
        "c4_en": stats_c4_en,
    },
    "frequencies": {
        "wiki_pt": freq_wiki_pt,
        "wiki_en": freq_wiki_en,
        "mc4_pt": freq_mc4_pt,
        "c4_en": freq_c4_en,
    },
    "lsi": {
        "wiki": lsi_wiki,
        "web": lsi_web,
        "combined": lsi_combined,
    },
    "masks": {
        "active": active,
        "pt_both": pt_both,
        "pt_wiki_only": pt_wiki_only,
        "pt_web_only": pt_web_only,
    },
    "selected": {
        "pt": selected_pt,
        "cross": selected_cross,
        "en": selected_en,
        "all": all_selected,
    },
    "examples": {
        "pt": examples_pt,
        "cross": examples_cross,
        "en": examples_en,
    },
}

torch.save(results, "fase3_results.pt")
print("Resultados completos salvos em fase3_results.pt")
print()
print("Arquivos gerados:")
print("  fase3_results.pt              — todos os dados")
print("  stats_wiki_pt.pt              — checkpoint Wikipedia PT")
print("  stats_wiki_en.pt              — checkpoint Wikipedia EN")
print("  stats_mc4_pt.pt               — checkpoint Web-crawl PT")
print("  stats_c4_en.pt                — checkpoint Web-crawl EN")
print("  fig_fase3_overview.png        — 4 painéis (distribuições + triangulação + heatmap)")
print("  fig_fase3_heatmap_top50_pt.png — heatmap detalhado top-50 PT")

Resultados completos salvos em fase3_results.pt

Arquivos gerados:
  fase3_results.pt              — todos os dados
  stats_level2_wiki.pt          — checkpoint Wikipedia
  stats_level1_web.pt           — checkpoint Web-crawl
  fig_fase3_overview.png        — 4 painéis (distribuições + triangulação + heatmap)
  fig_fase3_heatmap_top50_pt.png — heatmap detalhado top-50 PT
